In [1]:
import tensorflow as tf
print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))


TensorFlow: 2.19.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
import os, cv2, zipfile
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models


In [4]:
zip_path = "/content/drive/MyDrive/Parkinson's Dataset Final.zip"
extract_path = "/content/dataset"

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(extract_path)

print("Dataset extracted")


Dataset extracted


In [5]:
IMG_SIZE = 224

def load_images(folder, label):
    X, y = [], []
    for root, _, files in os.walk(folder):
        for f in files:
            if f.lower().endswith(('.png','.jpg','.jpeg')):
                img = cv2.imread(os.path.join(root,f), cv2.IMREAD_GRAYSCALE)
                img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
                img = img.astype(np.float32) / 255.0
                X.append(img[..., np.newaxis])
                y.append(label)
    return X, y


In [6]:
X, y = [], []

# OLD dataset
X_h, y_h = load_images("/content/dataset/Parkinson's Dataset Final/Hand Pd", 0)
X_p, y_p = load_images("/content/dataset/Parkinson's Dataset Final/Hand Pd", 1)

# NEW dataset
X_h2, y_h2 = load_images("/content/dataset/Parkinson's Dataset Final/New hand pd Dataset/Healthy", 0)
X_p2, y_p2 = load_images("/content/dataset/Parkinson's Dataset Final/New hand pd Dataset/Parkinson's", 1)

X = np.array(X_h + X_p + X_h2 + X_p2)
y = np.array(y_h + y_p + y_h2 + y_p2)

print("Total images:", len(X))


Total images: 2128


In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)


In [8]:
class PatchEmbedding(layers.Layer):
    def __init__(self, patch_size=16, embed_dim=526):
        super().__init__()
        self.proj = layers.Conv2D(embed_dim, patch_size, patch_size)

    def call(self, x):
        x = self.proj(x)
        x = tf.reshape(x, (tf.shape(x)[0], -1, x.shape[-1]))
        return x


In [9]:
class PositionalEncoding(layers.Layer):
    def __init__(self, n_patches, dim):
        super().__init__()
        self.pos = layers.Embedding(n_patches, dim)

    def call(self, x):
        positions = tf.range(start=0, limit=tf.shape(x)[1])
        return x + self.pos(positions)


In [10]:
class TransformerBlock(layers.Layer):
    def __init__(self, dim, heads=8, mlp_ratio=4):
        super().__init__()
        self.attn = layers.MultiHeadAttention(heads, dim)
        self.norm1 = layers.LayerNormalization()
        self.norm2 = layers.LayerNormalization()
        self.mlp = models.Sequential([
            layers.Dense(dim*mlp_ratio, activation="gelu"),
            layers.Dense(dim)
        ])

    def call(self, x):
        x = self.norm1(x + self.attn(x,x))
        x = self.norm2(x + self.mlp(x))
        return x


In [11]:
def build_custom_vit():
    inputs = layers.Input((224,224,1))

    x = PatchEmbedding(16, 526)(inputs)
    x = PositionalEncoding(196, 526)(x)

    channel_plan = [526,256,128,64,32,16,8,4,2,1]

    for i in range(len(channel_plan)-1):
        x = TransformerBlock(channel_plan[i])(x)
        x = layers.Dense(channel_plan[i+1])(x)

    x = layers.LayerNormalization()(x)
    x = layers.GlobalAveragePooling1D()(x)

    return models.Model(inputs, x, name="Pure_ViT_Feature_Extractor")


In [12]:
vit = build_custom_vit()
vit.trainable = False
vit.summary()


Model: "Pure_ViT_Feature_Extractor"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 1)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ patch_embedding                 │ (None, None, 526)      │       135,182 │
│ (PatchEmbedding)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ positional_encoding             │ (None, None, 526)      │       103,096 │
│ (PositionalEncoding)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block               │ (None, None, 526)      │    11,084,924 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, None, 256)      │       134,912 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_1             │ (None, None, 256)      │     2,630,144 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, None, 128)      │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_2             │ (None, None, 128)      │       659,712 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, None, 64)       │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_3             │ (None, None, 64)       │       166,016 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, None, 32)       │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_4             │ (None, None, 32)       │        42,048 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, None, 16)       │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_5             │ (None, None, 16)       │        10,784 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, None, 8)        │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_6             │ (None, None, 8)        │         2,832 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_20 (Dense)                │ (None, None, 4)        │            36 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_7             │ (None, None, 4)        │           776 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_23 (Dense)                │ (None, None, 2)        │            10 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_8             │ (None, None, 2)        │           22

 Total params: 15,014,601 (57.28 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 15,014,601 (57.28 MB)

In [13]:
feature_extractor = tf.keras.Model(
    inputs=vit.input,
    outputs=vit.get_layer("transformer_block_7").output,
    name="ViT_Features_Block7"
)


In [14]:
X_train_feat = feature_extractor.predict(X_train, batch_size=16)
X_test_feat  = feature_extractor.predict(X_test, batch_size=16)

print(X_train_feat.shape)


107/107 ━━━━━━━━━━━━━━━━━━━━ 20s 107ms/step
27/27 ━━━━━━━━━━━━━━━━━━━━ 8s 295ms/step
(1702, 196, 4)


In [15]:
X_train_feat = X_train_feat.mean(axis=1)
X_test_feat  = X_test_feat.mean(axis=1)

print(X_train_feat.shape)


(1702, 4)


In [16]:
def build_fcnn_small(input_dim):
    inputs = tf.keras.Input(shape=(input_dim,))

    x = layers.Dense(32, activation="relu")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.2)(x)

    x = layers.Dense(8, activation="relu")(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)

    return tf.keras.Model(inputs, outputs)


In [17]:
fcnn = build_fcnn_small(X_train_feat.shape[1])

fcnn.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

history = fcnn.fit(
    X_train_feat, y_train,
    validation_data=(X_test_feat, y_test),
    epochs=100,
    batch_size=16,
    verbose=1
)


Epoch 1/100
107/107 ━━━━━━━━━━━━━━━━━━━━ 7s 21ms/step - accuracy: 0.5065 - loss: 0.6918 - val_accuracy: 0.4883 - val_loss: 0.7061
Epoch 2/100
107/107 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5434 - loss: 0.6892 - val_accuracy: 0.4883 - val_loss: 0.7017
Epoch 3/100
107/107 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5351 - loss: 0.6889 - val_accuracy: 0.4883 - val_loss: 0.7250
Epoch 4/100
107/107 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5360 - loss: 0.6878 - val_accuracy: 0.4883 - val_loss: 0.8488
Epoch 5/100
107/107 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5468 - loss: 0.6816 - val_accuracy: 0.4883 - val_loss: 1.1308
Epoch 6/100
107/107 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5532 - loss: 0.6807 - val_accuracy: 0.4883 - val_loss: 1.3233
Epoch 7/100
107/107 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5455 - loss: 0.6813 - val_accuracy: 0.4883 - val_loss: 1.7634
Epoch 8/100
107/107 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5585 - loss: 0.6818 - val_acc